In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/ML/cleaned_data.csv')
print(df.shape)
print(df.head())

(63675, 2)
   label                                           combined
0      1  LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1      1     Did they post their votes for Hillary already?
2      1  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3      0  Bobby Jindal, raised Hindu, uses story of Chri...
4      1  SATAN 2: Russia unvelis an image of its terrif...


In [ ]:
train_df, temp_df  = train_test_split(df, test_size=0.3, random_state=42)

In [ ]:
val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=42)

In [ ]:
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 44572
Validation: 9551
Test: 9552


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
sample = "This news is completely fake"
tokens = tokenizer(sample, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
print(tokens)
print(tokens['input_ids'].shape)

{'input_ids': tensor([[ 101, 2023, 2739, 2003, 3294, 8275,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,

In [ ]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
      text = self.texts.iloc[idx]
      encoding = self.tokenizer(
      text,
      max_length=self.max_length,
      truncation=True,
      padding='max_length',
      return_tensors='pt'
      )
      input_ids= encoding['input_ids'].squeeze()
      attention_mask=encoding['attention_mask'].squeeze()
      label=self.labels.iloc[idx]
      return input_ids, attention_mask, label

In [ ]:
train_dataset = FakeNewsDataset(
    texts=train_df['combined'],
    labels=train_df['label'],
    tokenizer=tokenizer
)
test_dataset = FakeNewsDataset(
    texts=test_df['combined'],
    labels=test_df['label'],
    tokenizer=tokenizer
)
val_dataset = FakeNewsDataset(
    texts=val_df['combined'],
    labels=val_df['label'],
    tokenizer=tokenizer
)

In [ ]:
print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 44572
Val: 9551
Test: 9552


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
batch = next(iter(train_loader))
input_ids, attention_mask, labels = batch
print(f"input_ids shape: {input_ids.shape}")
print(f"attention_mask shape: {attention_mask.shape}")
print(f"labels shape: {labels.shape}")

input_ids shape: torch.Size([32, 512])
attention_mask shape: torch.Size([32, 512])
labels shape: torch.Size([32])


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)

cuda


In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")

Epoch 1 Loss: 0.0518895668999796
Epoch 2 Loss: 0.013063163407717224
Epoch 3 Loss: 0.007063823621456616


In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, '/content/drive/MyDrive/ML/trained_model.pth')

print("Model saved to Google Drive!")

Model saved to Google Drive!


In [ ]:
checkpoint = torch.load('/content/drive/MyDrive/ML/trained_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
model.to(device)
print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
  for batch in test_loader:
        input_ids, attention_mask, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

accuracy = accuracy_score(all_labels, all_predictions)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(classification_report(all_labels, all_predictions, target_names=['Real', 'Fake']))



Test Accuracy: 54.64%
              precision    recall  f1-score   support

        Real       0.55      1.00      0.71      5219
        Fake       0.00      0.00      0.00      4333

    accuracy                           0.55      9552
   macro avg       0.27      0.50      0.35      9552
weighted avg       0.30      0.55      0.39      9552



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
sample = "BREAKING: Donald Trump has secretly signed an executive order to ban all social media platforms in the United States. Sources close to the White House confirm the order will take effect next week. The mainstream media is hiding this from the public."
tokens = tokenizer(sample, max_length=512, truncation=True, padding='max_length', return_tensors='pt')

input_ids = tokens['input_ids'].to(device)
attention_mask = tokens['attention_mask'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    predictions = torch.argmax(outputs.logits, dim=1)
    probabilities = torch.softmax(outputs.logits, dim=1)
    confidence = probabilities[0][predictions.item()].item() * 100


label="Fake" if predictions.item()==0 else "Real"
print(f"Predicted Label: {label}")
print(f"Confidence: {confidence:.2f}%")


Predicted Label: Real
Confidence: 99.98%


In [ ]:
print(df['label'].value_counts())
print(df[df['label']==0]['combined'].head(2))
print(df[df['label']==1]['combined'].head(2))

label
0    34790
1    28885
Name: count, dtype: int64
3     Bobby Jindal, raised Hindu, uses story of Chri...
11    May Brexit offer would hurt, cost EU citizens ...
Name: combined, dtype: object
0    LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1       Did they post their votes for Hillary already?
Name: combined, dtype: object


In [ ]:
sample = "After years of investigation, FBI sources confirm that Barack Obama was not born in the United States. Documents obtained exclusively show his birth certificate was forged. The deep state has been covering this up since 2008."
tokens = tokenizer(sample, max_length=512, truncation=True, padding='max_length', return_tensors='pt')

input_ids = tokens['input_ids'].to(device)
attention_mask = tokens['attention_mask'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    predictions = torch.argmax(outputs.logits, dim=1)
    probabilities = torch.softmax(outputs.logits, dim=1)
    confidence = probabilities[0][predictions.item()].item() * 100


label="Fake" if predictions.item()==0 else "Real"
print(f"Predicted Label: {label}")
print(f"Confidence: {confidence:.2f}%")

Predicted Label: Real
Confidence: 98.46%


In [ ]:
sample_text = test_df['combined'].iloc[0]
sample_label = test_df['label'].iloc[0]

tokens = tokenizer(sample_text, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
input_ids = tokens['input_ids'].to(device)
attention_mask = tokens['attention_mask'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    prediction = torch.argmax(outputs.logits, dim=1)
    probabilities = torch.softmax(outputs.logits, dim=1)

print(f"Actual label: {sample_label}")
print(f"Predicted: {prediction.item()}")
print(f"Probabilities: Fake={probabilities[0][0].item()*100:.2f}%, Real={probabilities[0][1].item()*100:.2f}%")
print(f"Text: {sample_text[:200]}")

Actual label: 0
Predicted: 0
Probabilities: Fake=100.00%, Real=0.00%
Text: After Irma ravages Havana, city highlights housing replacement drive HAVANA (Reuters) - After Hurricane Irma wrought havoc on Havana s decrepit buildings and killed four in building collapses there, c


In [ ]:
fake_samples = test_df[test_df['label'] == 0].head(5)

for idx, row in fake_samples.iterrows():
    tokens = tokenizer(row['combined'], max_length=512, truncation=True, padding='max_length', return_tensors='pt')
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        prediction = torch.argmax(outputs.logits, dim=1)

    print(f"Actual: {row['label']} | Predicted: {prediction.item()} | Text: {row['combined'][:100]}")

Actual: 0 | Predicted: 0 | Text: After Irma ravages Havana, city highlights housing replacement drive HAVANA (Reuters) - After Hurric
Actual: 0 | Predicted: 1 | Text: Dear GOP: Hire me and I’ll give you the debate of your dreams I write to you in your capacity as deb
Actual: 0 | Predicted: 0 | Text: Former assistant accuses exiled Chinese tycoon of rape in lawsuit BEIJING (Reuters) - A former perso
Actual: 0 | Predicted: 0 | Text: Watch: SNL Pays Tribute to Obama - ’Don’t Go!’ - Breitbart “Saturday Night Live” ended its first sho
Actual: 0 | Predicted: 0 | Text: Canada and Denmark Fight Over Island With Whisky and Schnapps - The New York Times International dis


In [ ]:
real_samples = test_df[test_df['label'] == 1].head(3)
for idx, row in real_samples.iterrows():
    print(f"Label 1: {row['combined'][:150]}")
    print("---")

Label 1: COLIN POWELL SAYS HILLARY LYING About Private Email Server Conversation: “has no recollection of the dinner conversation” [VIDEO] For anyone who s pay
---
Label 1:  WATCH: Donald Trump GOES OFF On Fraud Victims Whose Lives Were Ruined By His Fake University (VIDEO) Donald Trump pretty much goes ballistic anytime 
---
Label 1: BREAKING: FIVE DAYS LATE! OBAMA SHAMED INTO ORDERING FLAGS AT HALF-STAFF FOR VICTIMS OF CHATTANOOGA TERROR ATTACK It s about time! Did Obama order the
---


In [ ]:
sample = "BREAKING: Scientists discover that the moon is actually made of cheese. NASA has been hiding this secret for decades."

tokens = tokenizer(sample, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
input_ids = tokens['input_ids'].to(device)
attention_mask = tokens['attention_mask'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    prediction = torch.argmax(outputs.logits, dim=1)
    probabilities = torch.softmax(outputs.logits, dim=1)
    confidence = probabilities[0][prediction.item()].item() * 100

label = "Real" if prediction.item() == 0 else "Fake"
print(f"Prediction: {label}")
print(f"Confidence: {confidence:.2f}%")

Prediction: Fake
Confidence: 99.99%


In [ ]:
def predict(text):
    tokens = tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        prediction = torch.argmax(outputs.logits, dim=1)
        probabilities = torch.softmax(outputs.logits, dim=1)
        confidence = probabilities[0][prediction.item()].item() * 100

    label = "Real" if prediction.item() == 0 else "Fake"
    return {"prediction": label, "confidence": round(confidence, 2)}

print(predict("BREAKING: Scientists discover that the moon is actually made of cheese!"))
print(predict("Scientists at NASA successfully launched a new climate monitoring satellite."))

{'prediction': 'Real', 'confidence': 57.26}
{'prediction': 'Real', 'confidence': 56.81}


In [ ]:
sample = "INDIANAPOLIS—Contending that the proposed restrictions were “common sense” and necessary to preserve the integrity of the sport, Indiana Fever shooting guard Sophie Cunningham claimed Friday that any woman who averages more than eight points per game should not be allowed to play in the WNBA. “Women who average more than 8.4 points, 2.3 rebounds, or 1.3 assists per game clearly have an unfair biological advantage over women who don’t, and we need to maintain a level playing field,” said Cunningham at a press conference, arguing that single-digit scorers shouldn’t have to worry about losing opportunities to players who possessed superior shooting, passing, and rebounding skills. “I don’t care what kind of criticism I get for saying this, but we need to protect women’s sports from people who are good at women’s sports. My concern is for the lower-to-average-tier players in the locker room who don’t have anyone else speaking up for them. If I get flack for saying what we’re all thinking, then so be it.” Cunningham added that she would never apologize for defending the next generation of girls from one day having to compete against someone who can reliably create her own shot."
print(predict(sample))

{'prediction': 'Real', 'confidence': 98.33}


In [ ]:
sample = """The U.S. government plans to take a 35% passive stake in Venezuelan businessman Alejandro Betancourt’s North American ​Blue Energy Partners, the Wall Street Journal reported ‌on Saturday, citing people involved in negotiating the deal.
The U.S. also plans to secure preferential rights to purchase 20% of the ​company's production at cost, the WSJ reported.
Jumpstart your morning with the latest legal news delivered straight to your inbox from The Daily Docket newsletter. Sign up here.
The Pentagon's ​Office of Strategic Capital plans to structure the investment ⁠through penny warrants that would yield the U.S. ​an equity ownership in the business without significant capital investment, ​the WSJ reported.
Advertisement · Scroll to continue
The report comes only a day after U.S. President Donald Trump said on Friday that the U.S. would take control of ​a fifth of Venezuela's vast oil reserves.
Trump provided few details ​about the arrangement, saying only that the U.S. had secured majority ‌control ⁠of more than 65 billion barrels of Venezuela's proven oil reserves through a partnership with private business.
Responding to a Reuters request for comment, chief Pentagon spokesperson Sean Parnell ​said: "The Office of ​Strategic Capital (OSC) ⁠does not take equity stakes in private companies. Under its statutory authority, OSC's role ​is strictly limited to providing capital assistance ​in the ⁠form of a loan, loan guarantee, or technical assistance (including transaction structuring for developing and financing investments)."""
print(predict(sample))

{'prediction': 'Real', 'confidence': 86.88}


In [ ]:
sample = """An MIT Media Lab study used EEG to track what happens in the brain when people write with ChatGPT rather than on their own. Fifty four students completed essay tasks while wearing electrode caps. One group used ChatGPT, one used a search engine, and one used no tools. Across sessions, the ChatGPT group showed the weakest neural connectivity in networks linked to memory, attention, and analytical work. They also had more trouble recalling what they had just written and produced more generic prose. The authors describe this as a kind of cognitive debt: the tool does the synthesizing, and the brain stays quieter [1].

The same experiment points to a better order of use. Students who first wrote unaided and only later switched to ChatGPT showed stronger recall and more widespread brain activity than students who started with the model and then had to work alone. In other words, the model helped more when it scaffolded thinking that was already underway than when it replaced that thinking from the start. The practical lesson is not that large language models should be banned. It is that they work best as a second pass, after the writer has already done the hard part of forming and remembering an argument [1].

[Kosmyna, N., Hauptmann, E., Yuan, Y. T., Situ, J., Liao, X. H., Beresnitzky, A. V., Braunstein, I., & Maes, P. (2025). Your Brain on ChatGPT: Accumulation of Cognitive Debt when Using an AI Assistant for Essay Writing Task. arXiv. DOI: 10.48550/arXiv.2506.08872]"""
print(predict(sample))

{'prediction': 'Real', 'confidence': 57.77}


In [ ]:
model.save_pretrained('/content/drive/MyDrive/ML/vera_model')
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
tokenizer.save_pretrained('/content/drive/MyDrive/ML/vera_tokenizer')
print('Model saved in hugging face format')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved in hugging face format


In [ ]:
sample = """Rockstar Leaks Worsen As Hackers Release Fully Playable Copy Of ‘GTA IX’
EDINBURGH, SCOTLAND—Sending the publisher scrambling to protect trade secrets related to the hotly anticipated project, leaks from Rockstar Games worsened Friday as hackers released a fully playable copy of Grand Theft Auto IX. “It’s heartbreaking having assets leak out for any of our projects, but that’s particularly the case when the grav-room immersion technology needed to fuse your somatosensory cortex with the world of Neo-Liberty City won’t exist for nearly 70 years,” read a statement from the company that urged players to wait until the game’s planned Nov. 19, 2096, release, when they could experience GTA IX ’s groundbreaking visuals, rich world, and gameplay mechanics in the intended form of a hypercube that vibrates at para-Newtonian frequencies. “So much of what’s out there is incomplete and unintelligible without knowing the wider context. For example, the game’s dialogue is written entirely in a Sino-Anglo-Swahili patois that really won’t become the global language until the 2070s. So, as tempting as it might be, we urge players to wait to experience the pulse-pounding crime story and romance between protagonists Grum-Gruck and the android Brvuaxxr-13 until the game finally comes out on the PlayStation 12.” Despite the additional pressure, Rockstar Games stressed that the studio continued to have no plans for a physical release of GTA IX, which it said would only be available to players who dissolved their neuro-ghost into a digital universe known as the Nano Stream."""
print(predict(sample))

{'prediction': 'Real', 'confidence': 55.79}


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle

In [ ]:
print("Training LR model...")
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train = tfidf.fit_transform(train_df['combined'])
X_val = tfidf.transform(val_df['combined'])

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, train_df['label'])

preds = lr_model.predict(X_val)
print(f"LR Accuracy: {accuracy_score(val_df['label'], preds)*100:.2f}%")

Training LR model...
LR Accuracy: 95.22%


In [ ]:
pickle.dump(lr_model, open('/content/drive/MyDrive/ML/lr_model.pkl', 'wb'))
pickle.dump(tfidf, open('/content/drive/MyDrive/ML/tfidf.pkl', 'wb'))
print("LR model and TF-IDF saved!")

LR model and TF-IDF saved!
